# SpectralDINO Open Set Recognition (OSR) Benchmark on Kaggle

This notebook runs the unified **Open Set Recognition (OSR)** benchmark for **SpectralDINO** (*Dual Mixture-of-Subspaces Low-Rank Adaptation for Cross-Domain Hyperspectral Image Few-Shot Classification*).

### Pipeline Structure:
1. **Houston Cross-Scene**: Houston 2013 (Source) $\rightarrow$ Houston 2018 (Target)
   - Step 1: Pre-train SpectralDINO (Dual-MoS LoRA ViT) + 384-dim Feature Extraction
   - Step 2: Closed-Set Few-Shot Baseline (Zero-Shot & Adaptation, 1/5/10 shots)
   - Step 3: Open Set Recognition (MSP Novelty Scoring, Macro AUROC & Known Acc across 20 splits $\times$ 5 seeds)
2. **HyRank Cross-Scene**: Dioni (Source) $\rightarrow$ Loukia (Target)
   - Step 1: Pre-train SpectralDINO (Dual-MoS LoRA ViT) + 384-dim Feature Extraction
   - Step 2: Closed-Set Few-Shot Baseline (Zero-Shot & Adaptation, 1/5/10 shots)
   - Step 3: Open Set Recognition (MSP Novelty Scoring, Macro AUROC & Known Acc across 84 splits $\times$ 5 seeds)


In [ ]:
# Check GPU and CUDA availability
!nvidia-smi

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device:      {torch.cuda.get_device_name(0)}")


In [ ]:
# Step 1: Workspace Setup, Code Deployment & Dataset Discovery
import os, shutil, glob, zipfile, subprocess

working_dir = "/kaggle/working"
input_dir = "/kaggle/input"
target_dir = os.path.join(working_dir, "SpectralDINO-Open-Set")
cs_run_py = os.path.join(target_dir, "cross-scene", "run.py")

print("=== Checking /kaggle/input ===")
for item in glob.glob("/kaggle/input/**/*", recursive=False):
    print(" -", item)

# 1. Ensure SpectralDINO-Open-Set code is properly deployed
if not os.path.exists(cs_run_py):
    print("Deploying SpectralDINO-Open-Set code...")
    found_code = False
    
    # 1a. Check for uncompressed folders in input or working dir
    for search_root in [input_dir, working_dir]:
        if not os.path.isdir(search_root):
            continue
        for root, dirs, files in os.walk(search_root):
            if root.startswith(target_dir):
                continue
            if "run.py" in files and "step1_extract_features.py" in files and ("cross-scene" in root.lower() or "spectraldino" in root.lower()):
                src_code_root = os.path.dirname(root) if os.path.basename(root) in ["cross-scene", "hyrank"] else root
                print(f"Copying code from {src_code_root} -> {target_dir}")
                shutil.copytree(src_code_root, target_dir, dirs_exist_ok=True)
                found_code = True
                break
        if found_code:
            break
            
    # 1b. Check for zip files (e.g., SpectralDINO-Open-Set.zip)
    if not os.path.exists(cs_run_py):
        for search_root in [input_dir, working_dir]:
            if not os.path.isdir(search_root):
                continue
            for root, dirs, files in os.walk(search_root):
                for f in files:
                    if f.lower().endswith(".zip") and ("spectraldino" in f.lower() or "open-set" in f.lower()):
                        zpath = os.path.join(root, f)
                        try:
                            with zipfile.ZipFile(zpath, 'r') as zf:
                                if any("run.py" in n for n in zf.namelist()):
                                    print(f"Extracting code archive: {zpath} -> {working_dir}")
                                    zf.extractall(working_dir)
                                    if not os.path.exists(cs_run_py):
                                        for cand in glob.glob(f"{working_dir}/**/cross-scene/run.py", recursive=True):
                                            cand_dir = os.path.dirname(os.path.dirname(cand))
                                            shutil.copytree(cand_dir, target_dir, dirs_exist_ok=True)
                                            break
                                    found_code = True
                                    break
                        except Exception:
                            pass
                if found_code:
                    break

    # 1c. Direct Git clone fallback from GitHub
    if not os.path.exists(cs_run_py):
        print("Cloning directly from GitHub: https://github.com/yoginlangalia/HyperJEPA_OSR_Project.git ...")
        tmp_repo = "/kaggle/working/_repo_tmp"
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/yoginlangalia/HyperJEPA_OSR_Project.git", tmp_repo], check=False)
        src = os.path.join(tmp_repo, "SpectralDINO", "SpectralDINO-Open-Set")
        if os.path.exists(src):
            shutil.copytree(src, target_dir, dirs_exist_ok=True)
            print("Successfully deployed SpectralDINO-Open-Set from GitHub!")
        shutil.rmtree(tmp_repo, ignore_errors=True)

os.makedirs(os.path.join(target_dir, "cross-scene", "data"), exist_ok=True)
os.makedirs(os.path.join(target_dir, "hyrank", "data"), exist_ok=True)

if os.path.exists(cs_run_py):
    print("SUCCESS: SpectralDINO-Open-Set code is ready at:", target_dir)
else:
    print("Error: Could not locate SpectralDINO-Open-Set code!")

# 2. Auto-detect and extract all dataset files and archives
cs_data_dir = os.path.join(target_dir, "cross-scene", "data")
hr_data_dir = os.path.join(target_dir, "hyrank", "data")

def auto_deploy_datasets():
    for search_root in [input_dir, working_dir]:
        if not os.path.isdir(search_root):
            continue
        for root, dirs, files in os.walk(search_root):
            if root.startswith(cs_data_dir) or root.startswith(hr_data_dir):
                continue
            for f in files:
                f_lower = f.lower()
                f_path = os.path.join(root, f)
                if f_lower.endswith('.zip'):
                    try:
                        with zipfile.ZipFile(f_path, 'r') as zf:
                            names = [n.lower() for n in zf.namelist()]
                            if any('houston' in n for n in names):
                                print(f"Auto-extracting Houston dataset: {f_path} -> {cs_data_dir}")
                                zf.extractall(cs_data_dir)
                            if any(k in names for k in ['dioni.mat', 'loukia.mat', 'hyrank']):
                                print(f"Auto-extracting HyRank dataset: {f_path} -> {hr_data_dir}")
                                zf.extractall(hr_data_dir)
                    except Exception:
                        pass
                elif f_lower in ['houston13.mat', 'houston13_7gt.mat', 'houston18.mat', 'houston18_7gt.mat']:
                    dst = os.path.join(cs_data_dir, f)
                    if not os.path.exists(dst):
                        print(f"Copying {f_path} -> {dst}")
                        shutil.copy2(f_path, dst)
                elif f_lower in ['dioni.mat', 'dioni_gt_out68.mat', 'dioni_gt.mat', 'loukia.mat', 'loukia_gt_out68.mat', 'loukia_gt.mat']:
                    dst = os.path.join(hr_data_dir, f)
                    if not os.path.exists(dst):
                        print(f"Copying {f_path} -> {dst}")
                        shutil.copy2(f_path, dst)

auto_deploy_datasets()

print("\n=== Dataset Status ===")
h13_found = os.path.exists(os.path.join(cs_data_dir, 'Houston13.mat'))
dioni_found = os.path.exists(os.path.join(hr_data_dir, 'Dioni.mat'))
print(f"Houston Cross-Scene Data Ready: {h13_found} ({cs_data_dir})")
print(f"HyRank Cross-Scene Data Ready:  {dioni_found} ({hr_data_dir})")
if not h13_found:
    print("Notice: If Houston13.mat is not ready, click + Add Input in sidebar to attach Houston-cross-scene-data.zip!")


---
## 2. Houston Cross-Scene (Houston 2013 $\rightarrow$ Houston 2018)
Runs:
- **Step 1**: Pretrain SpectralDINO on H13 + 384-dim Feature Extraction (GPU)
- **Step 2**: Closed-Set Few-Shot SVM baseline
- **Step 3**: Open Set Recognition (OSR) evaluation (MSP novelty scoring, Macro AUROC & Known Accuracy across 20 splits and 5 seeds)


In [ ]:
%cd /kaggle/working/SpectralDINO-Open-Set/cross-scene
!python run.py


In [ ]:
# Display Houston OSR Results
res_path = "CrossScene/CrossScene_OSR_results.txt"
if os.path.exists(res_path):
    print(open(res_path).read())
else:
    print("Results file not found yet.")


---
## 3. HyRank Cross-Scene (Dioni $\rightarrow$ Loukia)
Runs:
- **Step 1**: Pretrain SpectralDINO on Dioni + 384-dim Feature Extraction (GPU)
- **Step 2**: Closed-Set Few-Shot SVM baseline
- **Step 3**: Open Set Recognition (OSR) evaluation (MSP novelty scoring, Macro AUROC & Known Accuracy across 84 splits and 5 seeds)


In [ ]:
%cd /kaggle/working/SpectralDINO-Open-Set/hyrank
!python run.py


In [ ]:
# Display HyRank OSR Results
res_path = "HyRank/HyRank_OSR_results.txt"
if os.path.exists(res_path):
    print(open(res_path).read())
else:
    print("Results file not found yet.")


---
## 4. Package Results for Download


In [ ]:
%cd /kaggle/working
!mkdir -p OSR_Results_Summary_SpectralDINO
!cp -f SpectralDINO-Open-Set/cross-scene/CrossScene/CrossScene_OSR_results.txt OSR_Results_Summary_SpectralDINO/ || true
!cp -f SpectralDINO-Open-Set/cross-scene/CrossScene/CrossScene_results.txt OSR_Results_Summary_SpectralDINO/ || true
!cp -f SpectralDINO-Open-Set/hyrank/HyRank/HyRank_OSR_results.txt OSR_Results_Summary_SpectralDINO/ || true
!cp -f SpectralDINO-Open-Set/hyrank/HyRank/HyRank_results.txt OSR_Results_Summary_SpectralDINO/ || true
!zip -r OSR_Results_Summary_SpectralDINO.zip OSR_Results_Summary_SpectralDINO/
print("Results packaged in /kaggle/working/OSR_Results_Summary_SpectralDINO.zip")

from IPython.display import FileLink
display(FileLink('OSR_Results_Summary_SpectralDINO.zip'))
